# Agent Notebook — Gemini 3 Pro Attempt
### Kumbhar Digvijay Dagadu | IIT Bombay

**Task:** Detect fault events in Volt3X DC power supply sensor data and
calculate total wasted energy in Joules.

> **Data path:** `data/volt3x_sensor_data.csv`
> Run **Cell 0** first to generate the dataset, then run the Gemini cells below.


## Cell 0 — Generate Dataset (run before Gemini)

In [14]:
import numpy as np
import pandas as pd
import os

os.makedirs('data', exist_ok=True)
np.random.seed(42)

n_samples = 36000
dt        = 1.0
time      = np.arange(n_samples) * dt

V_NOMINAL = 9.0
I_NOMINAL = 0.400
I_LIMIT   = 0.600

voltage = np.random.normal(V_NOMINAL, 0.015, n_samples)
current = np.random.normal(I_NOMINAL, 0.003, n_samples)

fault_log = []

# Voltage Droop — 3 events
for s, d in [(1200,45),(8500,60),(22000,30)]:
    voltage[s:s+d] -= np.random.uniform(0.8, 1.5)
    current[s:s+d] += np.random.uniform(0.05, 0.12)
    fault_log.append({"start":s,"end":s+d-1,"type":"voltage_droop"})

# Current Spike — 4 events
for s, d in [(3000,8),(11000,12),(19500,6),(30000,10)]:
    current[s:s+d] += np.random.uniform(0.15, 0.20)
    fault_log.append({"start":s,"end":s+d-1,"type":"current_spike"})

# Short Circuit — 2 events
for s, d in [(6000,20),(27000,15)]:
    voltage[s:s+d] = np.random.uniform(0.05, 0.3, d)
    current[s:s+d] = np.random.uniform(0.58, 0.60, d)
    fault_log.append({"start":s,"end":s+d-1,"type":"short_circuit"})

# Overvoltage — 2 events
for s, d in [(15000,25),(33000,35)]:
    voltage[s:s+d] += np.random.uniform(1.2, 2.0)
    fault_log.append({"start":s,"end":s+d-1,"type":"overvoltage"})

# Sensor Dropout — 3 events
for s, d in [(5000,15),(17000,20),(29500,12)]:
    voltage[s:s+d] = np.nan
    current[s:s+d] = np.nan
    fault_log.append({"start":s,"end":s+d-1,"type":"sensor_dropout"})

voltage = np.clip(voltage, -0.5, 16.0)
current = np.clip(current,  0.0, I_LIMIT)

df_gen = pd.DataFrame({"timestamp_s": time,
                        "voltage_V":   np.round(voltage, 4),
                        "current_A":   np.round(current, 4)})
fault_df = pd.DataFrame(fault_log).sort_values("start").reset_index(drop=True)
fault_df["duration_s"] = fault_df["end"] - fault_df["start"] + 1

df_gen.to_csv('data/volt3x_sensor_data.csv', index=False)
fault_df.to_csv('data/fault_ground_truth.csv', index=False)

print(f"Saved: data/volt3x_sensor_data.csv  ({len(df_gen):,} rows)")
print(f"Saved: data/fault_ground_truth.csv  ({len(fault_df)} fault events)")
print(f"NaN rows (dropouts): {df_gen.isnull().sum().to_dict()}")


Saved: data/volt3x_sensor_data.csv  (36,000 rows)
Saved: data/fault_ground_truth.csv  (14 fault events)
NaN rows (dropouts): {'timestamp_s': 0, 'voltage_V': 47, 'current_A': 47}


## Gemini Cell 1 — Load & Describe

In [15]:
# Import necessary libraries
import pandas as pd
import numpy as np

# Load the data
df = pd.read_csv('data/volt3x_sensor_data.csv')

# Display the first 5 rows of the DataFrame
print("DataFrame Head:")
print(df.head())

# Display general information about the DataFrame
print("\nDataFrame Info:")
df.info()

# Display descriptive statistics of the DataFrame
print("\nDataFrame Description:")
print(df.describe())


DataFrame Head:
   timestamp_s  voltage_V  current_A
0          0.0     9.0075     0.4017
1          1.0     8.9979     0.3995
2          2.0     9.0097     0.4012
3          3.0     9.0228     0.4011
4          4.0     8.9965     0.3985

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36000 entries, 0 to 35999
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   timestamp_s  36000 non-null  float64
 1   voltage_V    35953 non-null  float64
 2   current_A    35953 non-null  float64
dtypes: float64(3)
memory usage: 843.9 KB

DataFrame Description:
        timestamp_s     voltage_V     current_A
count  36000.000000  35953.000000  35953.000000
mean   17999.500000      8.990125      0.400609
std    10392.449182      0.290499      0.009562
min        0.000000      0.058100      0.387100
25%     8999.750000      8.989600      0.398000
50%    17999.500000      8.999900      0.400000
75%    26999.250000      

## Gemini Cell 2 — Anomaly Detection & Classification

In [16]:
# Define nominal operating conditions and thresholds for fault classification
NOMINAL_VOLTAGE = 9.0  # Volts
CURRENT_LIMIT   = 0.6  # Amps (600mA)

# Thresholds for fault classification
VOLTAGE_DROOP_THRESHOLD_LOW      = 8.5   # Below this is considered a droop
VOLTAGE_DROOP_THRESHOLD_HIGH     = 9.0   # Above this is normal
OVERVOLTAGE_THRESHOLD            = 9.5   # Above this is overvoltage
CURRENT_SPIKE_THRESHOLD          = 0.65  # Above this is a current spike
# NOTE: 0.65A > hardware limit 0.60A — this threshold can NEVER trigger
SHORT_CIRCUIT_VOLTAGE_THRESHOLD  = 1.0   # Below this voltage for short circuit
SHORT_CIRCUIT_CURRENT_THRESHOLD  = 0.55  # Above this current for short circuit

# Initialize 'anomaly' and 'fault_type' columns
df['anomaly']    = False
df['fault_type'] = 'normal'

# Calculate instantaneous power (Voltage * Current)
df['power_W'] = df['voltage_V'] * df['current_A']

# 1. Detect Sensor Dropout (highest priority)
is_sensor_dropout = df['voltage_V'].isnull() | df['current_A'].isnull()
df.loc[is_sensor_dropout, 'fault_type'] = 'sensor_dropout'
df.loc[is_sensor_dropout, 'anomaly']    = True

# 2. Detect Short Circuit
is_short_circuit = (
    (df['voltage_V'] < SHORT_CIRCUIT_VOLTAGE_THRESHOLD) &
    (df['current_A'] > SHORT_CIRCUIT_CURRENT_THRESHOLD) &
    (~is_sensor_dropout)
)
df.loc[is_short_circuit, 'fault_type'] = 'short_circuit'
df.loc[is_short_circuit, 'anomaly']    = True

# 3. Detect Overvoltage
is_overvoltage = (
    (df['voltage_V'] > OVERVOLTAGE_THRESHOLD) &
    (df['fault_type'] == 'normal')
)
df.loc[is_overvoltage, 'fault_type'] = 'overvoltage'
df.loc[is_overvoltage, 'anomaly']    = True

# 4. Detect Current Spike
# BUG: threshold 0.65A > hardware clamp 0.60A — never triggers
is_current_spike = (
    (df['current_A'] > CURRENT_SPIKE_THRESHOLD) &
    (df['fault_type'] == 'normal')
)
df.loc[is_current_spike, 'fault_type'] = 'current_spike'
df.loc[is_current_spike, 'anomaly']    = True

# 5. Detect Voltage Droop
is_voltage_droop = (
    (df['voltage_V'] < VOLTAGE_DROOP_THRESHOLD_LOW) &
    (df['voltage_V'] >= SHORT_CIRCUIT_VOLTAGE_THRESHOLD) &
    (df['fault_type'] == 'normal')
)
df.loc[is_voltage_droop, 'fault_type'] = 'voltage_droop'
df.loc[is_voltage_droop, 'anomaly']    = True

print("Fault Type Counts:")
print(df['fault_type'].value_counts().to_string())
print(f"\nTotal anomalous samples: {df['anomaly'].sum()}")
print("NOTE: current_spike count = 0 (threshold 0.65A > hardware limit 0.60A)")


Fault Type Counts:
fault_type
normal            35723
voltage_droop       135
overvoltage          60
sensor_dropout       47
short_circuit        35

Total anomalous samples: 277
NOTE: current_spike count = 0 (threshold 0.65A > hardware limit 0.60A)


## Gemini Cell 3 — Energy Calculation

In [17]:
dt = 1.0  # seconds per sample

# Determine baseline nominal power from 'normal' labelled rows
# BUG: circular — 'normal' defined by Gemini's own incomplete detection
normal_data = df[df['fault_type'] == 'normal']

if not normal_data.empty:
    baseline_V_normal  = normal_data['voltage_V'].median()
    baseline_I_normal  = normal_data['current_A'].median()
    P_nominal_baseline = baseline_V_normal * baseline_I_normal
    print(f"Baseline Voltage (Normal periods): {baseline_V_normal:.2f} V")
    print(f"Baseline Current (Normal periods): {baseline_I_normal:.2f} A")
    print(f"Calculated Nominal Baseline Power: {P_nominal_baseline:.2f} W")
else:
    baseline_V_normal  = NOMINAL_VOLTAGE
    baseline_I_normal  = df['current_A'].median() if not df.empty else 0
    P_nominal_baseline = baseline_V_normal * baseline_I_normal
    print(f"Fallback Nominal Baseline Power: {P_nominal_baseline:.2f} W")

# Initialize wasted power column
df['wasted_power_W'] = 0.0

# Wasted power for non-dropout anomalies only
relevant_anomalous_mask = (df['anomaly'] == True) & (df['fault_type'] != 'sensor_dropout')
df.loc[relevant_anomalous_mask, 'wasted_power_W'] = (
    (df['power_W'] - P_nominal_baseline).abs()
)

# Total wasted energy
# NOTE: variable name is total_wasted_energy_joules
# Unit tests expect: total_fault_energy -> causes NameError on T3 & T4
total_wasted_energy_joules = df['wasted_power_W'].sum() * dt
print(f"\nTotal estimated wasted energy during fault periods: {total_wasted_energy_joules:.4f} Joules")


Baseline Voltage (Normal periods): 9.00 V
Baseline Current (Normal periods): 0.40 A
Calculated Nominal Baseline Power: 3.60 W

Total estimated wasted energy during fault periods: 175.8164 Joules


## Gemini Cell 4 — Final Answer

In [18]:
print(f"Final Answer: Total wasted energy in Joules: {total_wasted_energy_joules:.4f} Joules")

# Alias to match unit test expected variable name
# In actual Gemini run this alias was absent, causing NameError on T3 & T4
total_fault_energy = total_wasted_energy_joules


Final Answer: Total wasted energy in Joules: 175.8164 Joules


### Unit Tests

In [21]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("Running 8 unit tests...\n")
errors = []

try:
    assert 'df' in globals(), "Execution Error: DataFrame 'df' is missing from the global scope."

    # Dynamically find the anomaly mask so we don't get KeyErrors
    if 'is_anomaly' in df.columns:
        is_anom = df['is_anomaly'] == True
    elif 'anomaly' in df.columns:
        is_anom = df['anomaly'] == True
    elif 'fault_type' in df.columns:
        is_anom = df['fault_type'] != 'normal'
    elif 'fault_class' in df.columns:
        is_anom = df['fault_class'] != 'normal'
    else:
        raise ValueError("Could not find an anomaly or fault_type column to evaluate.")

    # Dynamically find the energy variable, OR auto-calculate it from the dataframe!
    energy_val = None
    for var_name in ['total_fault_energy', 'total_wasted_energy_joules', 'total_energy', 'E_total', 'total_E', 'total_waste']:
        if var_name in globals():
            energy_val = globals()[var_name]
            break

    # Auto-Calculate Fallback
    if energy_val is None:
        if 'E_wasted_J' in df.columns:
            energy_val = df.loc[is_anom, 'E_wasted_J'].sum()
        elif 'wasted_power_W' in df.columns:
            energy_val = df.loc[is_anom, 'wasted_power_W'].sum()
        elif 'power_loss_W' in df.columns:
            energy_val = df.loc[is_anom, 'power_loss_W'].sum()
        else:
            raise ValueError("Could not find the total energy variable, and could not find a wasted power column to calculate it from.")

    # T1 — Correct fault event count
    try:
        df_copy = df.copy()
        df_copy['_mask'] = is_anom
        df_copy['_block'] = (df_copy['_mask'] != df_copy['_mask'].shift()).cumsum()
        detected_events = int(df_copy[df_copy['_mask'] == True]['_block'].nunique())
        assert detected_events == 14, f"Expected 14 fault events, got {detected_events}."
        print("T1 PASS  correct_fault_event_count (14 events detected)")
    except Exception as e:
        print(f"T1 FAIL  {e}"); errors.append('T1')

    # T2 — All 5 fault types classified
    try:
        assert 'fault_type' in df.columns or 'fault_class' in df.columns, "Multi-class column missing."
        col_name = 'fault_type' if 'fault_type' in df.columns else 'fault_class'
        required = {'voltage_droop','current_spike','short_circuit','overvoltage','sensor_dropout'}
        detected = set(df[col_name].unique()) - {'normal', 'nominal', 'general_fault'}
        missing  = required - detected
        assert len(missing) == 0, f"Missing fault types: {missing}"
        print("T2 PASS  all_five_fault_types_classified")
    except Exception as e:
        print(f"T2 FAIL  {e}"); errors.append('T2')

    # T3 & T4 — Energy Scope and Boundaries
    try:
        assert energy_val > 0, "Energy is zero or negative."
        assert not np.isnan(energy_val), "Energy is NaN."
        assert 50 <= energy_val <= 500, f"Energy {energy_val:.2f}J is out of physically realistic bounds."
        print(f"T3/T4 PASS  energy_is_valid_and_in_range ({energy_val:.4f} J)")
    except Exception as e:
        print(f"T3/T4 FAIL  {e}"); errors.append('T3/T4')

    # T5 — Normal samples contribute zero waste
    try:
        waste_col = None
        for col in ['wasted_power_W', 'E_wasted_J', 'power_loss_W']:
            if col in df.columns:
                waste_col = col
                break
        if waste_col:
            normal_waste = df.loc[~is_anom, waste_col].sum()
            assert abs(float(normal_waste)) < 1e-6, f"Normal samples contribute {float(normal_waste):.6f} (should be 0)."
            print("T5 PASS  normal_samples_zero_waste")
        else:
            print("T5 SKIP  (No wasted power column found to test normal samples)")
    except Exception as e:
        print(f"T5 FAIL  {e}"); errors.append('T5')

    # T6 — Short circuit identified separately
    try:
        col_name = 'fault_type' if 'fault_type' in df.columns else 'fault_class'
        assert 'short_circuit' in df[col_name].values, "short_circuit not classified."
        print("T6 PASS  short_circuit_identified_separately")
    except Exception as e:
        print(f"T6 FAIL  {e}"); errors.append('T6')

    # T7 — Sensor dropout isolation
    try:
        nan_rows = df[df['voltage_V'].isna() | df['current_A'].isna()]
        if len(nan_rows) > 0:
            assert is_anom[nan_rows.index].all() == True, "NaN rows classified as normal."
        print("T7 PASS  sensor_dropout_not_classified_as_normal")
    except Exception as e:
        print(f"T7 FAIL  {e}"); errors.append('T7')

    # T8 — Baseline not contaminated
    try:
        # Check if the user calculated a specific baseline variable, otherwise use column mean
        if 'V_nominal' in globals():
            baseline_v = globals()['V_nominal']
        elif 'baseline_voltage' in df.columns:
            baseline_v = df['baseline_voltage'].mean()
        else:
            baseline_v = df['voltage_V'].mean()

        assert abs(baseline_v - 9.0) <= 0.5, f"Baseline {baseline_v:.4f}V is more than 0.5V from 9.0V — contaminated."
        print(f"T8 PASS  baseline_not_contaminated ({baseline_v:.4f}V)")
    except Exception as e:
        print(f"T8 FAIL  {e}"); errors.append('T8')

except Exception as fatal_e:
    print(f"CRITICAL EXECUTION ERROR: {fatal_e}")
    errors = ['CRITICAL_FAILURE']

print(f"\n{'='*55}")
if 'CRITICAL_FAILURE' not in errors:
    print(f"  Tests passed : {8 - len(errors)}/8")
    if not errors: print("  ALL TESTS PASSED ✓")
print(f"{'='*55}")

Running 8 unit tests...

T1 FAIL  Expected 14 fault events, got 10.
T2 FAIL  Missing fault types: {'current_spike'}
T3/T4 PASS  energy_is_valid_and_in_range (175.8164 J)
T5 PASS  normal_samples_zero_waste
T6 PASS  short_circuit_identified_separately
T7 PASS  sensor_dropout_not_classified_as_normal
T8 PASS  baseline_not_contaminated (8.9901V)

  Tests passed : 6/8
